In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bryanpark/sudoku")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\bryanpark\sudoku\versions\3


In [3]:
import copy
import sudoku
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)
print(puzzle)
sudoku.display(puzzle)
print("givens:", givens)

Constructing a sudoku puzzle.
* creating the solution...
* constructing a puzzle...
_ _ 9 5 _ _ 3 _ 2
7 _ _ _ 1 _ 4 6 _
6 4 _ _ _ _ _ _ _
_ _ _ _ _ 6 _ 1 _
_ 1 4 7 3 _ _ _ _
2 7 _ 4 _ _ _ 8 _
_ 5 _ 6 4 _ 1 _ 7
_ 3 _ _ 8 _ _ 4 _
_ _ 1 9 _ 2 _ _ 5
[[0, 9, 0, 5, 2, 6, 7, 0, 4], [0, 0, 7, 9, 0, 0, 3, 0, 1], [8, 4, 2, 7, 0, 0, 9, 0, 0], [0, 5, 0, 0, 3, 0, 0, 0, 0], [3, 0, 0, 0, 8, 1, 0, 9, 0], [4, 0, 0, 2, 0, 0, 6, 0, 8], [0, 8, 1, 0, 9, 0, 0, 0, 5], [0, 7, 0, 8, 0, 4, 2, 0, 0], [2, 0, 0, 0, 0, 7, 0, 6, 0]]
_ 9 _ 5 2 6 7 _ 4
_ _ 7 9 _ _ 3 _ 1
8 4 2 7 _ _ 9 _ _
_ 5 _ _ 3 _ _ _ _
3 _ _ _ 8 1 _ 9 _
4 _ _ 2 _ _ 6 _ 8
_ 8 1 _ 9 _ _ _ 5
_ 7 _ 8 _ 4 2 _ _
2 _ _ _ _ 7 _ 6 _
givens: 36


In [4]:
import numpy as np
quizzes = np.zeros((1000000, 81), np.int32)
solutions = np.zeros((1000000, 81), np.int32)
for i, line in enumerate(open('data/sudoku.csv', 'r').read().splitlines()[1:]):
    quiz, solution = line.split(",")
    for j, q_s in enumerate(zip(quiz, solution)):
        q, s = q_s
        quizzes[i, j] = q
        solutions[i, j] = s
quizzes = quizzes.reshape((-1, 9, 9))
solutions = solutions.reshape((-1, 9, 9))

In [5]:
print(quizzes)

[[[0 0 4 ... 2 0 9]
  [0 0 5 ... 0 0 1]
  [0 7 0 ... 0 4 3]
  ...
  [6 0 0 ... 1 0 5]
  [0 0 3 ... 6 9 0]
  [0 4 2 ... 3 0 0]]

 [[0 4 0 ... 0 5 0]
  [1 0 7 ... 9 6 0]
  [5 2 0 ... 0 0 0]
  ...
  [0 9 0 ... 5 4 3]
  [6 0 0 ... 7 0 0]
  [2 5 0 ... 1 0 0]]

 [[6 0 0 ... 3 8 4]
  [0 0 8 ... 0 7 2]
  [0 0 0 ... 0 0 5]
  ...
  [3 1 0 ... 0 5 0]
  [0 8 9 ... 0 0 0]
  [5 0 2 ... 1 9 0]]

 ...

 [[0 0 0 ... 8 2 0]
  [0 6 1 ... 0 3 0]
  [0 5 0 ... 0 0 0]
  ...
  [0 0 7 ... 0 6 5]
  [0 0 0 ... 4 0 8]
  [0 8 6 ... 0 0 0]]

 [[0 7 0 ... 6 9 0]
  [0 0 3 ... 0 0 1]
  [0 0 0 ... 0 2 0]
  ...
  [0 0 0 ... 0 4 0]
  [0 5 1 ... 9 0 0]
  [9 4 0 ... 0 0 7]]

 [[3 0 0 ... 6 2 0]
  [1 0 0 ... 4 0 0]
  [0 0 5 ... 8 3 0]
  ...
  [4 8 0 ... 0 1 0]
  [2 0 3 ... 0 0 0]
  [0 7 0 ... 0 9 0]]]


In [6]:
df = pd.read_csv('data/sudoku.csv',dtype=str)
df

,quizzes,solutions
0,0043002090050090010700600430060020871900074000...,8643712593258497619712658434361925871986574322...
1,0401000501070039605200080000000000170009068008...,3461792581875239645296483719658324174729168358...
2,6001203840084590720000060050002640300700800069...,6951273841384596727248369158512647392739815469...
3,4972000001004000050000160986203000403009000000...,4972583161864397252537164986293815473759641828...
4,0059103080094030600275001000300002010008200070...,4659123781894735623275681497386452919548216372...
...,...,...
999995,3000280000290000300054001077402030980086070031...,3175289464291768356854391277462135989586472131...
999996,0030006000040860057000009409350407208067200502...,5234976811942863757685139429356417288167294532...
999997,0003508200618040300500090000700600029030070100...,7493568212618745393582197468749613529235876146...
999998,0702006900030400010000650205600300000947005800...,4752816936239478511893657245628341793947165828...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   quizzes    1000000 non-null  object
 1   solutions  1000000 non-null  object
dtypes: object(2)
memory usage: 15.3+ MB


In [8]:
# 81자리가 아닌 문제 확인 > 없음
df[df['quizzes'].str.len()!=81]

,quizzes,solutions


In [9]:
# 81자리가 아닌 답안지 확인 > 없음
df[df['solutions'].str.len()!=81]

,quizzes,solutions


입력(quiz) :  0~9 (0은 빈칸을 뜻함)  
정답(solution) : 1~9  
학습 라벨 : (solution - 1) → 0~8

In [10]:
quiz_str = df['quizzes'][0]
quiz = np.array([int(c) for c in quiz_str]).reshape(9,9).astype(np.int64)
quiz

array([[0, 0, 4, 3, 0, 0, 2, 0, 9],
       [0, 0, 5, 0, 0, 9, 0, 0, 1],
       [0, 7, 0, 0, 6, 0, 0, 4, 3],
       [0, 0, 6, 0, 0, 2, 0, 8, 7],
       [1, 9, 0, 0, 0, 7, 4, 0, 0],
       [0, 5, 0, 0, 8, 3, 0, 0, 0],
       [6, 0, 0, 0, 0, 0, 1, 0, 5],
       [0, 0, 3, 5, 0, 8, 6, 9, 0],
       [0, 4, 2, 9, 1, 0, 3, 0, 0]])

In [11]:
# AI가 이해할 수 있는 숫자 배열로 return
class SudokuDataset(Dataset):
    
    #  X: (10, 9, 9) float32 one-hot (0~9)
    #  y: (9, 9) int64  (0~8)  # 정답 1~9를 0~8로 shift
    #  mask: (9, 9) bool  # quiz==0 (빈칸 위치)
    
    def __init__(self, df: pd.DataFrame, quiz_col: str, sol_col: str):
        self.quiz = df[quiz_col].values
        self.sol  = df[sol_col].values

    def __len__(self):          # quiz 길이 - 없으니까 DataLoader 객체 생성시 오류남
        return len(self.quiz)   

    @staticmethod
    def _str_to_grid(s: str) -> np.ndarray:

        b = np.frombuffer(s.encode("ascii"), dtype=np.uint8) - ord("0")
        return b.reshape(9, 9).astype(np.int64)                 # 9x9형태로 변환해라

    # 한문제를 꺼낼 때마다 실행
    def __getitem__(self, idx: int):
        quiz_str = self.quiz[idx]
        sol_str  = self.sol[idx]

        quiz = self._str_to_grid(quiz_str)     # (9,9) 0~9
        sol  = self._str_to_grid(sol_str)      # (9,9) 1~9

        mask = (quiz == 0)                     # 마스크: 여기가 문제니까 여기를 풀어라
                                               # > [False False True] 이런식으로 채워서 True인 곳만 학습하게

        
        y = sol - 1                            # 라벨: 1~9 -> 0~8(ai한테는 이렇게 학습시켜야 이해 잘함)

        # 입력 원핫: 0~9 (10클래스)
        # X[d, i, j] = 1 if quiz[i,j] == d
        X = np.zeros((10, 9, 9), dtype=np.float32)
        for d in range(10):
            X[d] = (quiz == d)

        # 이해를 돕자면, 스도쿠판에서 숫자 0 ~ 9까지 한번씩 
        # 9 x 9 스도쿠판에 True False로 있냐 없냐를 적어두는 과정임 (ai가 이해하기 좋게)

        # torch 텐서로 변환
        X = torch.from_numpy(X)                 # float32   > 계산용이라 FLOAT
        y = torch.from_numpy(y.astype(np.int64))# int64     > 정답용이라 INT 
        mask = torch.from_numpy(mask)           # bool

        return X, y, mask

In [ ]:
dataset = SudokuDataset(df, "quizzes", "solutions")

loader = DataLoader(      
    dataset,
    batch_size=64,      # GPU, CPU는 2의 배수 연산에 최적화됨 그래서 2의 6승인 64 선택, 64 X 157 
    shuffle=True,
    num_workers=2,      # 윈도우면 0~2부터 시작 추천
    pin_memory=True
)
# 데이터로더 실제로 일어나는 일
# [디스크 / RAM] ──(데이터 읽기)──> [CPU] ──> [GPU]

# 여기서 num_workers가 데이터 읽기를 담당한대
# 0이면 메인 프로세스가 직접 데이터 로딩 * Windows 기본 추천
# 2면, 일꾼 2명이 미리 데이터 준비 * 리눅스 코랩이면 2 추천

X, y, mask = next(iter(loader))
print("X:", X.shape, X.dtype)           # (B, 10, 9, 9) torch.float32
print("y:", y.shape, y.dtype)           # (B, 9, 9) torch.int64
print("mask:", mask.shape, mask.dtype)  # (B, 9, 9) torch.bool
print("blank ratio in batch:", mask.float().mean().item())

X: torch.Size([64, 10, 9, 9]) torch.float32
y: torch.Size([64, 9, 9]) torch.int64
mask: torch.Size([64, 9, 9]) torch.bool
blank ratio in batch: 0.5823688507080078


c:\Users\Playdata\deep_learning\dl_venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
class SudokuCNN(nn.Module):
    def __init__(self, in_ch=10, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1),         # __init__ 에서 X[0].size(0) 이건 바깥변수를 참조하는거다보니 위험하다... 그래서 받아온 in_ch로 진행
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),        # 주변 칸들을 보고 새로운 특징을 만든다
            nn.ReLU(),
            nn.Conv2d(
                hidden,                                     # 들어오는 특징 개수
                hidden,                                     # 새로만들 특징 개수
                3,                                          # 3×3 창으로 봄
                                                            # □ □ □
                                                            # □ X □   ← 이 칸을 예측할 때 3x3 임 더 키울 수 있음 그치만 스도쿠는 가장 근처를 보고 하는게 맞음
                                                            # □ □ □
                padding=1                                   # 크기 유지 스도쿠는 9 x 9 판을 유지해야해서 크기 유지를 시키기 위함임.
            ),
            nn.ReLU(),
            nn.Conv2d(hidden, 9, 1)                         # 칸마다 9개 중 하나 고르는 분류
        )

    def forward(self, x):
        return self.net(x)                                  # 입력을 모델에 그대로 흘려보낸다
    
    
                                                            # x
                                                            # ↓
                                                            # Conv2d
                                                            # ↓
                                                            # ReLU
                                                            # ↓
                                                            # Conv2d
                                                            # ↓            > 이걸 모델에 그대로 흘려보낸다라고 한다
                                                            # ReLU
                                                            # ↓
                                                            # Conv2d
                                                            # ↓
                                                            # ReLU
                                                            # ↓
                                                            # Conv2d (logits)


## conv2d를 한 이유는

작은 필터(예: 3×3)를  
전체 격자에 공유해서  
이웃 패턴을 인식  
주변 칸을 보면서 규칙을 학습하는 연산

이런 구조가 스도쿠와 비슷하지 않을까?

In [ ]:
# 스도쿠에서 빈칸만 골라서 점수(손실)를 매기는 함수
def masked_ce_loss(logits, y, mask):
    # logits :   모델이 낸 예측 점수   logits = [2.3, -1.1, 0.5, 3.8, 0.2, -0.9, 1.0, -2.4, 0.1] 이런식으로 맞을 확률을 점수로 매김
    #  y	 :   정답
    # mask	 :   빈칸 표시(True/False)
    
    B = y.size(0)
    
    
#   원래 구조 (B,C,H,W)
#   (B, 9, 9, 9)
#   = (문제수, 클래스, 행, 열)
    
    # (B,C,H,W) -> (B*H*W, C)
    logits = logits.permute(0, 2, 3, 1).reshape(B*81, 9)
    y = y.reshape(B*81)
    mask = mask.reshape(B*81)

    logits_m = logits[mask]     # mask로 빈칸이었던(실제 문제였던) 부분만 남기고 없앤다음 그걸로 점수 매길예정
    y_m = y[mask]

    
    if logits_m.numel() == 0: # 빈칸이 하나도 없는 배치면 안전 처리 
        return torch.tensor(
            0.0                             # loss = 0
            , device=logits.device          # GPU/CPU 위치 맞춤
            , requires_grad=True            # 역전파 가능
        )

    return F.cross_entropy(logits_m, y_m)
# 모델의 예측 점수(logits)와
# 정답(y)을 비교해서
# 얼마나 틀렸는지 계산하는 함수

In [ ]:
import time

device = torch.device("cpu")

model = SudokuCNN(in_ch=10, hidden=64).to(device)           # 스도쿠 CNN 모델 생성 후 CPU로 이동

# 모델 가중치 
# #--------------------------------------------------------------
model.load_state_dict(                                      # 미리 학습해 둔 가중치 불러오기
    torch.load("data/sudoku_model.pt", map_location=device)
)
model.eval()                                                # 평가 모드로 전환
#--------------------------------------------------------------


opt = torch.optim.Adam(model.parameters(), lr=1e-3)       # 옵티마이저 정의

model.train()                                             # 학습 모드로 전환
for epoch in range(1, 16):  # 일단 16epoch만
    t0 = time.time()
    total_loss = 0.0
    total_cells = 0
    correct_cells = 0

    for X, y, mask in loader:                                 # 미니배치 단위로 데이터 반복 처리
        X = X.to(device)
        y = y.to(device)
        mask = mask.to(device)                                # 데이터를 모델과 같은 장치(CPU/GPU)로 이동

        opt.zero_grad()                                       # 이전 배치의 gradient 초기화 (누적 방지)
        logits = model(X)  # (B, 9, 9, 9) (B,C,H,W)           # 입력을 모델에 흘려보내서 예측 점수(logits) 생성

        loss = masked_ce_loss(logits, y, mask)                # 위의 빈칸이었던것만 모아서 점수 매겨서 손실 나옴
        loss.backward()                                       # 역전파
        opt.step()                                            # 계산된 gradient로 모델 가중치 업데이트

        total_loss += loss.item()                             # epoch 전체 평균 loss 계산을 위해 누적




        # 빈칸 cell accuracy (모니터링용)                 
        with torch.no_grad():                                 # 정확도 계산 시 gradient 추적 비활성화 (속도 + 메모리 절약)
            pred = logits.argmax(dim=1)  # (B, 9, 9)   > logits = [2.3, -1.1, 0.5, 3.8, 0.2, -0.9, 1.0, -2.4, 0.1] 형태가 정답지 숫자로 바뀌는 순간
            m = mask
            total_cells += m.sum().item()                     # 이번 배치의 빈칸 개수를 전체 빈칸 수에 누적
            correct_cells += ((pred == y) & m).sum().item()   # 빈칸 중에서 맞춘 칸만 세어서 정확도 계산

    dt = time.time() - t0                                     # epoch 하나 도는 데 걸린 시간 계산
    acc = (correct_cells / total_cells) if total_cells > 0 else 0.0       # 빈칸 기준 cell accuracy 계산 (0으로 나누는 것 방지)
    print(f"epoch {epoch} | loss {total_loss/len(loader):.4f} | blank_acc {acc:.4f} | time {dt:.1f}s")


SudokuCNN(
  (net): Sequential(
    (0): Conv2d(10, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(64, 9, kernel_size=(1, 1), stride=(1, 1))
  )
)

In [ ]:
import numpy as np
import torch
import copy

solution = sudoku.construct_puzzle_solution()                       # 완성된 스도쿠 정답판(9×9)을 하나 생성
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)        # 정답판에서 숫자 30개만 남기고 나머지를 지운 문제(puzzle) 생성

quiz = np.array(puzzle, dtype=np.int64)  # (9,9)                    
mask = (quiz == 0)

# one-hot (10,9,9)
X = np.zeros((10, 9, 9), dtype=np.float32)
for d in range(10):
    X[d] = (quiz == d)

# torch 텐서: (1,10,9,9)
a = torch.from_numpy(X).unsqueeze(0)  # add batch dim               # NumPy → Torch Tensor 변환 후 배치 차원 추가

model.eval()                                                        # 평가모드
with torch.no_grad():                                               # 예측 중에는 기울기 계산 하면 안돼
    logits = model(a)                 # (1,9,9,9) = (B,C,H,W)       
    pred = logits.argmax(dim=1)[0]    # (9,9) 0~8                   # 정답 맞추기

pred_1to9 = (pred + 1).cpu().numpy()  # (9,9)                       # 모델 출력(0~8)을 실제 숫자(1~9)로 복원 -> CPU로 옮기고 NumPy로 변환

filled = quiz.copy()                                                # 원래 문제판을 복사해서 결과 보드 생성
filled[mask] = pred_1to9[mask]                                      # 빈칸(mask=True) 위치만 모델 예측으로 채움

print("QUIZ:")
print(quiz)
print("\nFILLED:")
print(filled)


# => 실패

QUIZ:
[[7 0 6 9 5 2 0 0 4]
 [9 4 0 0 0 0 0 0 7]
 [0 0 3 8 0 0 0 1 0]
 [0 9 2 4 0 5 1 0 0]
 [0 0 0 0 0 0 0 0 3]
 [4 0 0 1 3 6 0 8 0]
 [2 5 0 0 6 1 0 4 0]
 [0 0 7 0 9 0 3 6 5]
 [0 0 0 0 7 8 2 0 0]]

FILLED:
[[7 1 6 9 5 2 8 6 4]
 [9 4 8 1 1 1 5 2 7]
 [2 5 3 8 4 4 5 1 8]
 [6 9 2 4 8 5 1 6 6]
 [1 3 6 9 8 4 6 5 3]
 [4 7 8 1 3 6 7 8 2]
 [2 5 9 3 6 1 7 4 8]
 [1 1 7 2 9 4 3 6 5]
 [3 4 1 3 7 8 2 9 1]]


# 아래서부터는 코테수준...

In [ ]:
import numpy as np
import torch

# 1. 보드 <-> 모델 입력 변환
def board_to_X(board: np.ndarray) -> torch.Tensor:
    # board: (9,9) int, 0=blank, 1~9=digits
    # return: (1,10,9,9) float32 one-hot (channel 0 = blank)
    X = np.zeros((10, 9, 9), dtype=np.float32)
    for d in range(10):
        X[d] = (board == d).astype(np.float32)
    return torch.from_numpy(X).unsqueeze(0)                   # (1,10,9,9) 형태로 고정


# 2. 특정 칸 (r,c)에 스도쿠 규칙상 들어갈 수 있는 숫자(1~9) 를 계산
def valid_candidates(board: np.ndarray, r: int, c: int) -> np.ndarray:
    if board[r, c] != 0:
        return np.zeros(9, dtype=bool)                          # 이미 채워진 칸이므로 전부 False 반환

    used = set(board[r, :]) | set(board[:, c])                  # 같은행과 열에 쓰인 숫자를 모아서 used에 담음
    br, bc = (r // 3) * 3, (c // 3) * 3                         
    used |= set(board[br:br + 3, bc:bc + 3].reshape(-1))        # 같은 박스 안에 있는 숫자들까지 합침
    used.discard(0)                                             # 이미 쓰인 숫자는 후보에서 제거

    mask = np.ones(9, dtype=bool)
    for v in used:                                              # 이미 쓰인 숫자는 후보에서 제거
        if 1 <= v <= 9:
            mask[v - 1] = False
    return mask


# CNN 예측을 스도쿠 규칙으로 걸러냄
def build_valid_mask(board: np.ndarray) -> np.ndarray:
    m = np.zeros((9, 9, 9), dtype=bool)
    for r in range(9):
        for c in range(9):
            if board[r, c] == 0:
                m[r, c] = valid_candidates(board, r, c)
    return m


# 3. 모델 출력 형태 통일
def logits_to_9x9x9(logits: torch.Tensor) -> torch.Tensor:
    
    # 모델이 어떤 형태로 내놓든 (1,9,9,9)로 통일.  
    if logits.dim() == 4 and logits.shape[-1] == 9:
        # (B,9,9,9)
        return logits
    if logits.dim() == 3 and logits.shape[1] == 81 and logits.shape[2] == 9:
        # (B,81,9) -> (B,9,9,9)
        return logits.view(logits.shape[0], 9, 9, 9)
    if logits.dim() == 3 and logits.shape[1] == 9 and logits.shape[2] == 81:
        # (B,9,81) -> (B,9,9,9)
        return logits.permute(0, 2, 1).contiguous().view(logits.shape[0], 9, 9, 9)
    raise ValueError(f"Unsupported logits shape: {tuple(logits.shape)}")


# 4. Autoregressive / Iterative Greedy Solver
@torch.no_grad()
def solve_iterative_greedy(
    model,
    board: np.ndarray,
    device: str = None,
    max_steps: int = 200,
    temperature: float = 1.0,
    return_trace: bool = False,
):
    
    #  핵심 아이디어
    #  1. 모델이 전체 칸 확률 예측
    #  2. 빈 칸 중 '가장 확신 높은' (r,c,값) 1개 선택
    #  3. 그 값 확정 후 보드 업데이트
    #  4. 반복
    # - constraint masking 포함: 불가능 숫자는 확률 0 처리

    model.eval()
    bd = board.copy().astype(np.int64)

    if device is None:
        # 모델 파라미터가 있는 device로 자동 추정
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
    else:
        device = torch.device(device)

    trace = []

    for step in range(max_steps):
        blanks = np.argwhere(bd == 0)
        if len(blanks) == 0:
            return (bd, trace) if return_trace else bd  # solved

        # (1,10,9,9)
        X = board_to_X(bd).to(device)

        logits = model(X)
        logits = logits_to_9x9x9(logits)  # (1,9,9,9)
        logits = logits[0]  # (9,9,9)

        # temperature
        if temperature != 1.0:
            logits = logits / float(temperature)

        # constraint mask: 불가능 숫자 -inf 처리
        valid = build_valid_mask(bd)  # (9,9,9) bool (numpy)                            # 스도쿠 규칙을 활용해서 들어갈 수 있는 숫자 체크
        valid_t = torch.from_numpy(valid).to(device)

        # 빈칸이 아닌 곳은 선택 대상에서 제외하려고, 일단 logits는 마스크만 적용
        masked_logits = logits.clone()
        masked_logits[~valid_t] = -1e9  # 불가능 digit 제거

        probs = torch.softmax(masked_logits, dim=-1)  # (9,9,9)

        # 빈칸 위치만 평가해서 "가장 확신 높은" 채우기 선택
        best_r, best_c, best_d = None, None, None
        best_p = -1.0

        for (r, c) in blanks:
            # 해당 칸에서 가장 높은 후보
            pvals = probs[r, c]  # (9,)
            p_max, d_idx = torch.max(pvals, dim=-1)                                         # argmax처럼 값으로 바꾸는 과정
            p_max = float(p_max.item())
            d_idx = int(d_idx.item())  # 0..8

            # 만약 후보가 전부 막혔으면(=0), 모순 상태
            if p_max <= 0.0:
                # 모순: 해결 실패
                return (None, trace) if return_trace else None

            if p_max > best_p:                                                              # 반복문 돌면서 가장 높은 한 칸 찾음
                best_p = p_max
                best_r, best_c, best_d = int(r), int(c), d_idx + 1  # digit 1..9            # 행 / 열 / 답

        # 선택한 값 확정
        bd[best_r, best_c] = best_d
        if return_trace:
            trace.append((best_r, best_c, best_d, best_p))

    # max_steps 초과: 아직 미해결
    return (None, trace) if return_trace else None


# 5. 사용
board = np.array(quiz, dtype=np.int64)  # (9,9), 0=blank
solved = solve_iterative_greedy(model, board, device="cpu", max_steps=300)
if solved is None:
    print("실패(모순 또는 max_steps 초과)")
else:
    print(solved)


[[7 1 6 9 5 2 8 3 4]
 [9 4 8 6 1 3 5 2 7]
 [5 2 3 8 4 7 6 1 9]
 [3 9 2 4 8 5 1 7 6]
 [8 6 1 7 2 9 4 5 3]
 [4 7 5 1 3 6 9 8 2]
 [2 5 9 3 6 1 7 4 8]
 [1 8 7 2 9 4 3 6 5]
 [6 3 4 5 7 8 2 9 1]]
